# ST-OMR Meter V4-4 — Final Holdout BBox Annotation

**Annotation only.** Bu notebook model/checkpoint açmaz ve inference çalıştırmaz.

Her bbox **tam meter işaretini** kapsamalıdır: üst rakam + alt rakam birlikte. Preview üzerinde çizim yapılır; kaydedilen koordinatlar original `image.png` integer pixel koordinatlarıdır.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import shutil, subprocess, sys
from pathlib import Path
EXPECTED_CODE_SHA = '8818253ecc280e90c51f0d1df77863c29267c783'
REPO_DIR = Path('/content/st-omr-meter-v4-4/repo')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR.parent)
REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
subprocess.run(['git','clone','--filter=blob:none','https://github.com/khfy7wpr5p-maker/st-omr-training.git',str(REPO_DIR)], check=True)
subprocess.run(['git','-C',str(REPO_DIR),'checkout','--detach',EXPECTED_CODE_SHA], check=True)
HEAD = subprocess.check_output(['git','-C',str(REPO_DIR),'rev-parse','HEAD'], text=True).strip()
assert HEAD == EXPECTED_CODE_SHA, (HEAD, EXPECTED_CODE_SHA)
sys.path.insert(0, str(REPO_DIR))
print('V4-4 pinned code HEAD=', HEAD)


In [ ]:
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/TEST/METER_V1/03_FINAL_HOLDOUT_150')
MANIFEST = ROOT / 'FINAL_HOLDOUT_150_SELECTION.json'
assert ROOT.is_dir(), f'Missing holdout root: {ROOT}'
assert MANIFEST.is_file(), f'Missing frozen selection manifest: {MANIFEST}'
print('root=', ROOT)
print('manifest=', MANIFEST)


In [ ]:
from st_omr_training.meter_v4_4_final_holdout_bbox_annotation import AnnotationSession
PRECHECK = AnnotationSession(candidate_root=str(ROOT), manifest_path=str(MANIFEST))
print({
    'selected_count': len(PRECHECK.samples),
    'annotated_count': PRECHECK.annotated_count,
    'resume_index': PRECHECK.resume_index() + 1,
    'review_flag_count': len(PRECHECK.progress['review_flags']),
    'image_binding_sha256': PRECHECK.binding['image_binding_sha256'],
    'model_evaluated': False,
    'inference_count': 0,
    'candidate_checkpoint_opened': False,
})


In [ ]:
from st_omr_training.meter_v4_4_bbox_annotation_colab import launch_colab_annotation
SESSION = launch_colab_annotation(candidate_root=str(ROOT), manifest_path=str(MANIFEST))
print(f'Resume: {SESSION.annotated_count}/150; next index={SESSION.resume_index()+1}')


## Annotation bittikten sonra

UI `150 / 150` gösterdiğinde ve tüm review flag'leri çözüldüğünde aşağıdaki QA hücresini çalıştır. Bu hücre model çalıştırmaz; `FINAL_HOLDOUT_150_BBOX_COMPLETE.json` ve contact sheet'leri üretir. Contact sheet insan tarafından ayrıca görsel kontrol edilmeden V4-5'e geçilmez.


In [ ]:
from st_omr_training.meter_v4_4_final_holdout_bbox_annotation import (
    generate_review_contact_sheets, write_completion_receipt,
)
receipt_path = write_completion_receipt(candidate_root=str(ROOT), manifest_path=str(MANIFEST))
sheets = generate_review_contact_sheets(candidate_root=str(ROOT), manifest_path=str(MANIFEST))
print('MECHANICAL_QA=PASS')
print('completion_receipt=', receipt_path)
print('contact_sheets=', [str(p) for p in sheets])
print('HUMAN_VISUAL_REVIEW_REQUIRED=True')
print('MODEL_EVALUATED=False; INFERENCE_COUNT=0; CANDIDATE_CHECKPOINT_OPENED=False')
